# Feature engineering

## Read csv

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import joblib
import os


In [2]:
df_clean = pd.read_csv("../data/OpenAlex/openalex_ai_papers_cleaned.csv")
df_clean.head()

,id,title,publication_year,language,cited_by_count,referenced_works_count,fwci,authorship_count,countries_distinct_count,publication_type,is_oa,oa_status,citation_top_1_percent,citation_top_10_percent,keyword_count,primary_topic_score,first_year_citations,topic_id,topic_name
0,https://openalex.org/W1902237438,Effective Approaches to Attention-based Neural...,2015,en,8508,20,633.3691,3,1,proceedings-article,1,gold,1,1,7,0.9999,155,T10181,Natural Language Processing Techniques
1,https://openalex.org/W658020064,From Word Embeddings To Document Distances,2015,en,1518,47,164.8113,4,1,other,0,closed,1,1,11,0.9996,61,T10181,Natural Language Processing Techniques
2,https://openalex.org/W2251135946,Distant Supervision for Relation Extraction vi...,2015,en,1200,31,87.4904,4,1,proceedings-article,1,gold,1,1,11,0.9992,17,T10181,Natural Language Processing Techniques
3,https://openalex.org/W1491002046,An Introduction to Recursive Partitioning Usin...,2015,en,1030,7,105.2714,2,0,unknown,0,closed,1,1,14,0.9880,54,T10181,Natural Language Processing Techniques
4,https://openalex.org/W2250342921,chrF: character n-gram F-score for automatic M...,2015,en,935,10,36.6006,1,1,proceedings-article,1,gold,1,1,8,1.0000,14,T10181,Natural Language Processing Techniques


In [3]:
print(df_clean.shape)
print(df_clean.dtypes)

(293045, 19)
id                           object
title                        object
publication_year              int64
language                     object
cited_by_count                int64
referenced_works_count        int64
fwci                        float64
authorship_count              int64
countries_distinct_count      int64
publication_type             object
is_oa                         int64
oa_status                    object
citation_top_1_percent        int64
citation_top_10_percent       int64
keyword_count                 int64
primary_topic_score         float64
first_year_citations          int64
topic_id                     object
topic_name                   object
dtype: object


## Drop leaky features

Now we will drop the columns that are not used for modeling - leaky or non needed.

In [4]:
cols_to_drop = [
    'id',
    'title',
    'cited_by_count',
    'fwci',
    'citation_top_1_percent',
    'first_year_citations'
]

df_model = df_clean.drop(columns=cols_to_drop)
print(df_model.shape)
df_model.columns.tolist()

(293045, 13)


['publication_year',
 'language',
 'referenced_works_count',
 'authorship_count',
 'countries_distinct_count',
 'publication_type',
 'is_oa',
 'oa_status',
 'citation_top_10_percent',
 'keyword_count',
 'primary_topic_score',
 'topic_id',
 'topic_name']

## Language column

Let's put all non-English languages to category `other`

In [5]:
df_model['language'] = df_model['language'].where(
    df_model['language'] == 'en', other='other'
)

print(df_model['language'].value_counts())

language
en       279468
other     13577
Name: count, dtype: int64


Separate target

In [6]:
target = 'citation_top_10_percent'

X = df_model.drop(columns=[target])
y = df_model[target]

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Target distribution:\n{y.value_counts(normalize=True).round(3)}')

X shape: (293045, 12)
y shape: (293045,)
Target distribution:
citation_top_10_percent
0    0.82
1    0.18
Name: proportion, dtype: float64


Check for nulls

In [7]:
null_counts = X.isnull().sum()
null_counts = null_counts[null_counts > 0]

if len(null_counts) == 0:
    print('No nulls found — ready to split')
else:
    print('Nulls found:')
    print(null_counts)

No nulls found — ready to split


## Numerical/Categorical features

In [8]:
categorical_cols = [
    'publication_type',
    'oa_status',
    'topic_name',
    'language'
]

numerical_cols = [
    'publication_year',
    'authorship_count',
    'countries_distinct_count',
    'referenced_works_count',
    'keyword_count',
    'primary_topic_score',
    'is_oa'
]

print('Categorical columns:', len(categorical_cols))
print('Numerical columns:', len(numerical_cols))
print('Total features:', len(categorical_cols) + len(numerical_cols))

Categorical columns: 4
Numerical columns: 7
Total features: 11


## Train/Test split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) 
# stratify=y ensures the 82/18 class balance is preserved in both train and test sets.

In [10]:
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train distribution:\n{y_train.value_counts(normalize=True).round(3)}')
print(f'y_test distribution:\n{y_test.value_counts(normalize=True).round(3)}')

X_train shape: (234436, 12)
X_test shape:  (58609, 12)
y_train distribution:
citation_top_10_percent
0    0.82
1    0.18
Name: proportion, dtype: float64
y_test distribution:
citation_top_10_percent
0    0.82
1    0.18
Name: proportion, dtype: float64


## One-hot encode categoricals

In [11]:
# Initialise encoder — handle_unknown='ignore' encodes unseen categories as all zeros
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on train only to avoid data leakage
ohe.fit(X_train[categorical_cols])

# Transform both train and test using the fitted encoder
encoded_train = ohe.transform(X_train[categorical_cols])
encoded_test = ohe.transform(X_test[categorical_cols])

# Get the new column names from the encoder
encoded_col_names = ohe.get_feature_names_out(categorical_cols)

# Combine numerical columns with encoded categorical columns
X_train_encoded = pd.concat([
    X_train[numerical_cols].reset_index(drop=True),
    pd.DataFrame(encoded_train, columns=encoded_col_names)
], axis=1)

X_test_encoded = pd.concat([
    X_test[numerical_cols].reset_index(drop=True),
    pd.DataFrame(encoded_test, columns=encoded_col_names)
], axis=1)

print(f'X_train_encoded shape: {X_train_encoded.shape}')
print(f'X_test_encoded shape:  {X_test_encoded.shape}')
X_train_encoded.columns.tolist()

X_train_encoded shape: (234436, 30)
X_test_encoded shape:  (58609, 30)


['publication_year',
 'authorship_count',
 'countries_distinct_count',
 'referenced_works_count',
 'keyword_count',
 'primary_topic_score',
 'is_oa',
 'publication_type_journal-article',
 'publication_type_other',
 'publication_type_proceedings-article',
 'publication_type_thesis',
 'publication_type_unknown',
 'oa_status_bronze',
 'oa_status_closed',
 'oa_status_diamond',
 'oa_status_gold',
 'oa_status_green',
 'oa_status_hybrid',
 'topic_name_Anomaly Detection Techniques and Applications',
 'topic_name_Evolutionary Algorithms and Applications',
 'topic_name_Metaheuristic Optimization Algorithms Research',
 'topic_name_Natural Language Processing Techniques',
 'topic_name_Neural Networks and Applications',
 'topic_name_Privacy-Preserving Technologies in Data',
 'topic_name_Quantum Computing Algorithms and Architecture',
 'topic_name_Sentiment Analysis and Opinion Mining',
 'topic_name_Speech Recognition and Synthesis',
 'topic_name_Topic Modeling',
 'language_en',
 'language_other']

## Saving features

In [ ]:
# Create output directories if they don't exist
os.makedirs('../data/features', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save encoded feature matrices
X_train_encoded.to_csv('../data/features/X_train.csv', index=False)
X_test_encoded.to_csv('../data/features/X_test.csv', index=False)

# Save target vectors — reset index to match encoded feature matrices
y_train.reset_index(drop=True).to_csv('../data/features/y_train.csv', index=False)
y_test.reset_index(drop=True).to_csv('../data/features/y_test.csv', index=False)

# Save fitted encoder for use on new data in modelling and dashboard
joblib.dump(ohe, '../models/ohe.pkl')

print('Saved:')
print('  ../data/features/X_train.csv')
print('  ../data/features/X_test.csv')
print('  ../data/features/y_train.csv')
print('  ../data/features/y_test.csv')
print('  ../models/ohe.pkl')


Saved:
  ../data/features/X_train.csv
  ../data/features/X_test.csv
  ../data/features/y_train.csv
  ../data/features/y_test.csv
  ../models/ohe.pkl
Saved:
  ../data/features/X_train.csv
  ../data/features/X_test.csv
  ../data/features/y_train.csv
  ../data/features/y_test.csv
  ../models/ohe.pkl
